## Project environment

In [3]:
# Importing libraries
import pandas as pd
from pathlib import Path

# Project root (current working) directory
project_root = Path.cwd()

# Define project directory
data_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

## Load source data
[![Format](https://img.shields.io/badge/Format-CSV-orange)]()
[![Type](https://img.shields.io/badge/Type-Invoice%20Data-blue)]()
[![Source](https://img.shields.io/badge/Source-EU%20Open%20Data-lightgrey)](https://data.europa.eu/data/datasets/https-catalog-ale-se-store-1-resource-91?locale=en)

In [4]:
source_file = "public_spending.csv" # Source data filename
invoice_df = pd.read_csv(data_dir / source_file, sep=',') # Read CSV-file (comma separated)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\piete\\PycharmProjects\\SupplierSegmentation-Clustering-Github\\notebooks\\data\\raw\\public_spending.csv'

## Preview and explore the data

In [53]:
invoice_df.head() # View first records
invoice_df.info(verbose=True) # Summary information
invoice_df.columns # Column names

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5272 entries, 0 to 5271
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   avtal                0 non-null      float64
 1   belopp               5272 non-null   object 
 2   datum                5272 non-null   object 
 3   fakturanummer        5272 non-null   object 
 4   forvaltning          5272 non-null   object 
 5   grund                0 non-null      float64
 6   kommun_id            0 non-null      float64
 7   konto_nr             5272 non-null   int64  
 8   konto_text           5272 non-null   object 
 9   kopare               0 non-null      float64
 10  kopare_id            0 non-null      float64
 11  leverantor           5272 non-null   object 
 12  leverantor_id        5271 non-null   float64
 13  s_kod_nr             0 non-null      float64
 14  verifikationsnummer  5272 non-null   int64  
dtypes: float64(7), int64(2), object(6)
mem

Index(['avtal', 'belopp', 'datum', 'fakturanummer', 'forvaltning', 'grund',
       'kommun_id', 'konto_nr', 'konto_text', 'kopare', 'kopare_id',
       'leverantor', 'leverantor_id', 's_kod_nr', 'verifikationsnummer'],
      dtype='object')

## Rename columns

In [54]:
# Renaming column header names
invoice_df = invoice_df.rename(columns={
    "avtal": "contract_id",
    "belopp": "invoice_amount",
    "datum": "invoice_date",
    "fakturanummer": "invoice_number",
    "forvaltning": "department_name",
    "grund": "invoice_descr8iption",
    "kommun_id": "municipality_id",
    "konto_nr": "gl_account_number",
    "konto_text": "gl_account_description",
    "kopare": "buyer_name",
    "kopare_id": "buyer_id",
    "leverantor": "supplier_name",
    "leverantor_id": "supplier_id",
    "s_kod_nr": "category_code",
    "verifikationsnummer": "voucher_number"})

## Select relevant columns

In [55]:
### Selecting relevant columns
invoice_df = invoice_df[[
    "invoice_amount",
    "invoice_date",
    "invoice_number",
    "department_name",
    "gl_account_number",
    "gl_account_description",
    "supplier_name",
    "supplier_id"]]

## Handle missing values

In [56]:
# =============================================================================
# NOTE:
# Rows with missing supplier_id are removed, because supplier_id is required
# to aggregate the data on supplier-level and link invoices to a supplier.
# =============================================================================
invoice_df.isna().sum() # Check each column for missing values
invoice_df = invoice_df[invoice_df["supplier_id"].notna()] # Remove rows with missing supplier_id

## Convert data types

In [57]:
# Clean and convert invoice_amount to float
invoice_df["invoice_amount"] = (invoice_df["invoice_amount"]
                                .astype(str) # Convert to string
                                .str.replace(" ", "",regex=False) # Remove white spaces
                                .str.replace(",", ".",regex=False) # Replace , by . character
                                .astype(float) # Convert to float
                                .round(2)) # Round to 2 decimals
# Convert invoice_date to datetime
invoice_df["invoice_date"] = pd.to_datetime(invoice_df["invoice_date"], format='%Y-%m-%d')
# Convert supplier_id to integer
invoice_df["supplier_id"] = invoice_df["supplier_id"].astype(int)
# Convert supplier_name to string
invoice_df["supplier_name"] = invoice_df["supplier_name"].astype(str).str.upper()

## Save processed data

In [58]:
output_file = "invoice_df_processed.parquet" # Save as Parquet file
invoice_df.to_parquet(processed_dir / output_file, index=False)